In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-40s :: %(message)s'
)

# Inverse potential problem 

Operator that maps the shape of a homogeneous heat source to the heat flux measured at some
circle outside of the object. The heat distributions satisfies

$$
        \begin{cases}
            \Delta u = 1_K & \text{ in } \Omega \\
            u = 0          & \text{ on } \partial\Omega
        \end{cases}
        
$$
where $\partial\Omega$ is the measurement circle and $K$ is the heat source. The operator
maps the shape of the heat source to the Neumann data:
$$
        \partial K \mapsto \frac{\partial u}{\partial\nu}|_{\partial\Omega}.
$$

In [ ]:
from potential import Potential
from regpy.vecsps.curve import StarTrigDiscr
from regpy.solvers import RegularizationSetting
from regpy.hilbert import L2, Sobolev


#Forward operator
op = Potential(
    radius=1.3
)

setting = RegularizationSetting(op=op, penalty=Sobolev, data_fid=L2)


In [ ]:
#Exact data and Poission data
exact_solution = op.domain.sample(lambda t: np.sqrt(3*np.cos(t)**2+1)/2)
exact_data = op(exact_solution)
noise = op.codomain.randn()
noise = 0.01*setting.h_codomain.norm(exact_data)/setting.h_codomain.norm(noise) * noise
data = exact_data + noise

In [ ]:

#Initial guess
init = op.domain.sample(lambda t: 1)

from regpy.solvers.nonlinear.irgnm import IrgnmCG
from regpy.solvers.nonlinear.newton import NewtonCG
import regpy.stoprules as rules

#Solver: NewtonCG or IrgnmCG
solver = NewtonCG(
    setting, data, init = init,
        cgmaxit=50, rho=0.6
)

"""
solver = IrgnmCG(
    setting, data,
    regpar=10,
    regpar_step=0.8,
    init=init,
    cg_pars=dict(
        tol=1e-4
    )
)
"""

stoprule = (
    rules.CountIterations(100) +
    rules.Discrepancy(
        setting.h_codomain.norm, data,
        noiselevel = setting.h_codomain.norm(noise),
        tau=1.1
    )
)

#Plot function

fig, axs = plt.subplots(1, 2)
axs[0].set_title('Obstacle')
axs[1].set_title('Heat flux')

reco, reco_data = solver.run(stoprule)


axs[0].plot(*op.domain.eval_curve(exact_solution).curve[0])
axs[0].plot(*op.domain.eval_curve(reco).curve[0])

axs[1].plot(exact_data, label='exact')
axs[1].plot(reco_data, label='reco')
axs[1].plot(data, label='measured')
axs[1].legend()
axs[1].set_ylim(ymin=0)

plt.show()